In [ ]:
import os
import sys

import pandas as pd

sys.path.append(os.path.abspath("../"))

In [ ]:
def load_and_process_runs(base_path, prefix):
    runs = [f for f in os.listdir(base_path) if f.startswith(prefix) and f.endswith(".csv")]
    dfs = [pd.read_csv(os.path.join(base_path, f)) for f in sorted(runs)]  # sort for determinism
    return dfs


grayscale_path = "../../data/evaluation/cpn-grayscale"
yuyv_path = "../../data/evaluation/cpn-yuyv"

grayscale_runs = load_and_process_runs(grayscale_path, "run_")
yuyv_runs = load_and_process_runs(yuyv_path, "run_")

id_cols = [
    col
    for col in grayscale_runs[0].columns
    if col.startswith(("resolution", "encoder_architecture"))
]
print("id_cols:", id_cols)

metric_cols = [
    col
    for col in grayscale_runs[0].columns
    if col.startswith(("classifier", "encoder")) and "architecture" not in col
]

# Fix: align on id_cols before subtracting
diffs = []
for i in range(min(len(grayscale_runs), len(yuyv_runs))):
    df_g = grayscale_runs[i].set_index(id_cols).sort_index()
    df_y = yuyv_runs[i].set_index(id_cols).sort_index()

    # Sanity check: both runs must have the same configurations
    assert df_g.index.equals(df_y.index), (
        f"Run {i}: grayscale and yuyv have different id_cols combinations!\n{df_g.index}\nvs\n{df_y.index}"
    )

    diff = df_g.copy()
    diff[metric_cols] = df_g[metric_cols] - df_y[metric_cols]
    diffs.append(diff.reset_index())
    

# Means and stds
mean_grayscale = pd.concat(grayscale_runs).groupby(id_cols).mean(numeric_only=True).reset_index()
std_grayscale  = pd.concat(grayscale_runs).groupby(id_cols).std(numeric_only=True).reset_index()
mean_yuyv      = pd.concat(yuyv_runs).groupby(id_cols).mean(numeric_only=True).reset_index()
std_yuyv       = pd.concat(yuyv_runs).groupby(id_cols).std(numeric_only=True).reset_index()

# Diff the means — align on id_cols to be safe
mean_g_indexed = mean_grayscale.set_index(id_cols)
mean_y_indexed = mean_yuyv.set_index(id_cols)
std_g_indexed  = std_grayscale.set_index(id_cols)
std_y_indexed  = std_yuyv.set_index(id_cols)

common   = mean_g_indexed.index.intersection(mean_y_indexed.index)
mean_diff = (mean_g_indexed.loc[common] - mean_y_indexed.loc[common]).reset_index()

# std of difference of two independent variables: sqrt(std_g² + std_y²)
std_diff = (
    (std_g_indexed.loc[common] ** 2 + std_y_indexed.loc[common] ** 2) ** 0.5
).reset_index()

# Save
mean_grayscale.to_csv(f"{grayscale_path}/mean.csv", index=False)
std_grayscale.to_csv(f"{grayscale_path}/std.csv", index=False)
mean_yuyv.to_csv(f"{yuyv_path}/mean.csv", index=False)
std_yuyv.to_csv(f"{yuyv_path}/std.csv", index=False)
mean_diff.to_csv("../../data/evaluation/cpn-grayscale-diff-mean.csv", index=False)
std_diff.to_csv("../../data/evaluation/cpn-grayscale-diff-std.csv", index=False)

In [ ]:
mean_diff

### Build Latex Table


In [ ]:
mean_yuyv

In [ ]:
def arch_key(arch):
    if not isinstance(arch, str):
        raise TypeError(f"arch_key expected a string, got {type(arch)}: {arch!r}")
    parts = arch.split("_")
    return f"{parts[0]}_{parts[-1]}"


# e.g. "ires_16x16_v1" → "ires_v1", "conv_32x32_v1" → "conv_v1"

arch_mapping = {
    "ires_v1": "CPN$_1$",
    "ires_v2": "CPN$_2$",
    "conv_v1": "CPN$_3$",
}

# mean_df = mean_yuyv
# std_df = std_yuyv

# mean_df = mean_grayscale
# std_df = std_grayscale

mean_df = mean_diff
std_df = std_diff

rounding_precision_recall = 1
rounding_precision_euc = 1
rounding_std = 2


rows = []
for (resolution, arch), group in mean_df.groupby(["resolution", "encoder_architecture"]):
    std_row = std_df[
        (std_df["resolution"] == resolution) & (std_df["encoder_architecture"] == arch)
    ]

    def fmt_pct(mean_col, std_col=None, r_mean=rounding_precision_recall, r_std=rounding_std):
        m = round(group[mean_col].iloc[0] * 100, r_mean)
        s = round(std_row[std_col or mean_col].iloc[0] * 100, r_std)
        return f"${m:.{r_mean}f} \\pm {s:.{r_std}f}$"

    def fmt_euc(mean_col, std_col=None, r_mean=rounding_precision_euc, r_std=rounding_std):
        m = round(group[mean_col].iloc[0], r_mean)
        s = round(std_row[std_col or mean_col].iloc[0], r_std)
        return f"${m:.{r_mean}f} \\pm {s:.{r_std}f}$"

    arch_label = arch_mapping[arch_key(arch)]
    cells = " & ".join(
        [
            fmt_pct("encoder_recall_at_k_ball"),
            fmt_pct("encoder_recall_at_k_penaltyMark"),
            fmt_pct("encoder_recall_at_k_intersections"),
            fmt_euc("encoder_euclidean_error_ball"),
            fmt_euc("encoder_euclidean_error_penaltyMark"),
            fmt_euc("encoder_euclidean_error_intersections"),
        ]
    )
    formatted = f"& {arch_label} & {cells} \\\\"

    rows.append(
        {
            "resolution": resolution,
            "encoder_architecture": arch,
            "formatted_string": formatted,
        }
    )

table = pd.DataFrame(rows)
table["arch_order"] = table["encoder_architecture"].apply(lambda a: arch_mapping[arch_key(a)])
diff_table = table.sort_values(["resolution", "arch_order"]).drop(columns="arch_order")

for resolution in sorted(diff_table["resolution"].unique(), reverse=True):
    print(resolution)
    print(
        "\n".join(diff_table[diff_table["resolution"] == resolution]["formatted_string"].tolist())
    )


In [ ]:
# ── Switch ────────────────────────────────────────────────────────────────────
MODE = "grayscale"  # "grayscale" | "yuyv" | "diff"
SHOW_STD = True  # False → show mean only
# ─────────────────────────────────────────────────────────────────────────────

if MODE == "grayscale":
    mean_df = mean_grayscale
    std_df_a = std_grayscale  # shown as \pm
    std_df_b = None
elif MODE == "yuyv":
    mean_df = mean_yuyv
    std_df_a = std_yuyv
    std_df_b = None
elif MODE == "diff":
    mean_df = mean_diff
    std_df_a = std_diff  # shown as \pm on the left side
    std_df_b = None  # shown as \pm on the right side
else:
    raise ValueError(f"Unknown MODE: {MODE!r}")

rounding_precision_recall = 1
rounding_precision_euc = 1
rounding_std = 2


def arch_key(arch):
    if not isinstance(arch, str):
        raise TypeError(f"arch_key expected a string, got {type(arch)}: {arch!r}")
    parts = arch.split("_")
    return f"{parts[0]}_{parts[-1]}"


arch_mapping = {
    "ires_v1": "CPN$_1$",
    "ires_v2": "CPN$_2$",
    "conv_v1": "CPN$_3$",
}


def lookup_std(std_df, resolution, arch, col):
    return std_df[(std_df["resolution"] == resolution) & (std_df["encoder_architecture"] == arch)][
        col
    ].iloc[0]


def fmt_pct(col, resolution, arch, r_mean=rounding_precision_recall, r_std=rounding_std):
    m = round(
        mean_df[(mean_df["resolution"] == resolution) & (mean_df["encoder_architecture"] == arch)][
            col
        ].iloc[0]
        * 100,
        r_mean,
    )
    if not SHOW_STD:
        return f"${m:.{r_mean}f}$"
    sa = round(lookup_std(std_df_a, resolution, arch, col) * 100, r_std)
    if std_df_b is not None:
        sb = round(lookup_std(std_df_b, resolution, arch, col) * 100, r_std)
        return f"${m:.{r_mean}f} \\pm ({sa:.{r_std}f} , {sb:.{r_std}f})$"
    return f"${m:.{r_mean}f} \\pm {sa:.{r_std}f}$"


def fmt_euc(col, resolution, arch, r_mean=rounding_precision_euc, r_std=rounding_std):
    m = round(
        mean_df[(mean_df["resolution"] == resolution) & (mean_df["encoder_architecture"] == arch)][
            col
        ].iloc[0],
        r_mean,
    )
    if not SHOW_STD:
        return f"${m:.{r_mean}f}$"
    sa = round(lookup_std(std_df_a, resolution, arch, col), r_std)
    if std_df_b is not None:
        sb = round(lookup_std(std_df_b, resolution, arch, col), r_std)
        return f"${m:.{r_mean}f} \\pm ({sa:.{r_std}f}, {sb:.{r_std}f})$"
    return f"${m:.{r_mean}f} \\pm {sa:.{r_std}f}$"


rows = []
for (resolution, arch), _ in mean_df.groupby(["resolution", "encoder_architecture"]):
    arch_label = arch_mapping[arch_key(arch)]
    cells = " & ".join(
        [
            fmt_pct("encoder_recall_at_k_ball", resolution, arch),
            fmt_pct("encoder_recall_at_k_penaltyMark", resolution, arch),
            fmt_pct("encoder_recall_at_k_intersections", resolution, arch),
            fmt_euc("encoder_euclidean_error_ball", resolution, arch),
            fmt_euc("encoder_euclidean_error_penaltyMark", resolution, arch),
            fmt_euc("encoder_euclidean_error_intersections", resolution, arch),
        ]
    )
    rows.append(
        {
            "resolution": resolution,
            "encoder_architecture": arch,
            "formatted_string": f"& {arch_label} & {cells} \\\\",
        }
    )

table = pd.DataFrame(rows)
table["arch_order"] = table["encoder_architecture"].apply(lambda a: arch_mapping[arch_key(a)])
diff_table = table.sort_values(["resolution", "arch_order"]).drop(columns="arch_order")

print(f"\n=== MODE: {MODE} ===")
for resolution in sorted(diff_table["resolution"].unique(), reverse=True):
    print(resolution)
    print(
        ("\n".join(diff_table[diff_table["resolution"] == resolution]["formatted_string"].tolist())).replace(".", "{,}")
    )